# Spatial Analysis of Multiplexed Imaging Data

This notebook demonstrates the spatial analysis tools available in `oyLabImaging` for single-cell resolved microscopy data.  All methods operate on a `PosLbl` object (`P`) — a segmented position — and work with physical coordinates in **µm**.

## What's covered

| Section | Method | Question answered |
|---|---|---|
| Radial correlation | `P.radial_corr` | Are nearby cells more correlated in expression than distant ones? |
| Pair correlation | `P.radial_density` | Are cells clustered, dispersed, or randomly arranged? |
| Lengthscale fit | `P.fit_corr_lengthscale` | What is the characteristic distance of spatial correlation? |
| Mark variogram | `P.mark_variogram` | How quickly does expression similarity decay with distance? |
| LISA | `P.local_moran_I` / `P.plot_lisa` | Which individual cells are part of spatially autocorrelated clusters? |
| Gi* | `P.gistar` / `P.plot_gistar` | Where are expression hot spots and cold spots? |
| Spatial regions | `P.spatial_regions` / `P.plot_spatial_regions` | Does the tissue have distinct spatial niches? |
| GWR | `P.gwr` / `P.plot_gwr` | Do channel-to-channel correlations vary spatially? |

Results are cached automatically in `P.spatial` and persisted to disk so they survive kernel restarts.  Call `P.spatial['key']()` on any cached result to get a hint on how to retrieve or replot it.

In [1]:
%load_ext autoreload
%autoreload 2
%gui qt
%matplotlib qt5

import sys

#import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from oyLabImaging import Metadata

In [4]:
fpath = "/bigstore/pirlo/Images2026/Maddie/20260306_A549_overlay_pSTAT1_clusters"
MD = Metadata(fpath)

loaded MM metadata from /bigstore/pirlo/Images2026/Maddie/20260306_A549_overlay_pSTAT1_clusters/metadata.pickle


In [5]:
MD()

,acq,Position,frame,Channel,Marker,group,XY,Z,Zindex,Exposure,PixelSize,PlateType,TimestampFrame,TimestampImage,filename,root_pth
0,confocal/Acq1_1,B2-Site_0_0,0,DeepBlue,DeepBlue,None,"[23380.0, 19240.0]",4119.32,0,50.0,0.65,NA,2026-03-06 11:38:37.937 -0500,2026-03-06 11:38:37.937 -0500,/confocal/Acq1_1/B2-Site_0/img_channel000_posi...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
1,confocal/Acq1_1,B2-Site_0_0,0,Yellow,Yellow,None,"[23380.0, 19240.0]",4119.32,0,10.0,0.65,NA,2026-03-06 11:38:38.798 -0500,2026-03-06 11:38:38.798 -0500,/confocal/Acq1_1/B2-Site_0/img_channel001_posi...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
2,confocal/Acq1_1,B2-Site_0_0,0,FarRed,FarRed,None,"[23380.0, 19240.0]",4119.32,0,500.0,0.65,NA,2026-03-06 11:38:40.174 -0500,2026-03-06 11:38:40.174 -0500,/confocal/Acq1_1/B2-Site_0/img_channel002_posi...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
3,confocal/Acq1_1,B2-Site_1_0,0,DeepBlue,DeepBlue,None,"[23380.0, 21240.0]",4119.32,0,50.0,0.65,NA,2026-03-06 11:38:40.580 -0500,2026-03-06 11:38:40.580 -0500,/confocal/Acq1_1/B2-Site_1/img_channel000_posi...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
4,confocal/Acq1_1,B2-Site_1_0,0,Yellow,Yellow,None,"[23380.0, 21240.0]",4119.32,0,10.0,0.65,NA,2026-03-06 11:38:41.469 -0500,2026-03-06 11:38:41.469 -0500,/confocal/Acq1_1/B2-Site_1/img_channel001_posi...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
571,widefield/Acq1_1,G9-Site_0_1,0,Yellow,Yellow,None,"[86380.0, 64240.0]",4122.82,0,2.0,0.65,NA,2026-03-06 11:50:41.047 -0500,2026-03-06 11:50:41.047 -0500,/widefield/Acq1_1/G9-Site_0/img_channel001_pos...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
572,widefield/Acq1_1,G9-Site_0_1,0,FarRed,FarRed,None,"[86380.0, 64240.0]",4122.82,0,50.0,0.65,NA,2026-03-06 11:50:41.980 -0500,2026-03-06 11:50:41.980 -0500,/widefield/Acq1_1/G9-Site_0/img_channel002_pos...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
573,widefield/Acq1_1,G9-Site_1_1,0,DeepBlue,DeepBlue,None,"[86879.6, 65307.200000000004]",4122.82,0,5.0,0.65,NA,2026-03-06 11:50:42.311 -0500,2026-03-06 11:50:42.311 -0500,/widefield/Acq1_1/G9-Site_1/img_channel000_pos...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...
574,widefield/Acq1_1,G9-Site_1_1,0,Yellow,Yellow,None,"[86879.6, 65307.200000000004]",4122.82,0,2.0,0.65,NA,2026-03-06 11:50:43.192 -0500,2026-03-06 11:50:43.192 -0500,/widefield/Acq1_1/G9-Site_1/img_channel001_pos...,/bigstore/pirlo/Images2026/Maddie/20260306_A54...


In [6]:
if MD.test_duplicates():
    MD.fix_duplicate_posnames()
else:
    print('all good')

all good


## Setup — Metadata and Flat-Field Correction

`Metadata` is the entry point for an experiment.  It reads the acquisition log written by Micro-Manager and exposes every image as a row in a dataframe.

**Flat-field correction** removes the non-uniform illumination profile of the microscope (typically bright in the centre, dimmer toward the edges).  `MD.CalculateFlatField()` estimates the illumination field for each channel and acquisition by averaging across all positions and decomposing the result with an à-trous wavelet — separating the smooth, large-scale illumination pattern from actual biological signal.  The field is saved to `FlatFields/` and attached to `MD` so all subsequent image reads can apply the correction automatically with `ffield=True`.

> **When to use:** Always compute a flat field before extracting cell intensities if you care about quantitative measurements, especially near field edges.

In [7]:
MD.CalculateFlatField()

Flat field estimated for channel 'DeepBlue', acq 'confocal/Acq1_1' (50 images).
Flat field estimated for channel 'DeepBlue', acq 'widefield/Acq1_1' (50 images).
Flat field estimated for channel 'Yellow', acq 'confocal/Acq1_1' (50 images).
Flat field estimated for channel 'Yellow', acq 'widefield/Acq1_1' (50 images).
Flat field estimated for channel 'FarRed', acq 'confocal/Acq1_1' (50 images).
Flat field estimated for channel 'FarRed', acq 'widefield/Acq1_1' (50 images).


In [8]:
MD.save()

2026-05-21 19:41:55,388 [INFO] saved metadata


In [9]:
import napari

pos = MD.unique('Position')[20]
ch  = 'FarRed'
acq = MD.unique('acq', Position=pos)[0]

img_raw = MD.stkread(Position=pos, Channel=ch, ffield=False).squeeze()
img_ffc = MD.stkread(Position=pos, Channel=ch, ffield=True).squeeze()
ff      = MD._get_flatfield(ch, acq)

viewer = napari.Viewer(title=f'Flat-field correction | {ch} | {pos}')
viewer.add_image(img_raw, name='Raw',                    colormap='gray')
viewer.add_image(img_ffc, name='Flat-field corrected',   colormap='gray')
viewer.add_image(ff,      name='Illumination field',     colormap='inferno', opacity=0.8)

opening file img_channel002_position020_time000000000_z000.tif
Loaded group C4-Site_0_0 of images.
opening file img_channel002_position020_time000000000_z000.tif , applying flat field FlatFields/FarRed_confocal_Acq1_1.tif
Loaded group C4-Site_0_0 of images.


<Image layer 'Illumination field' at 0x7fa534f81d30>

## Segmentation and Loading

`results` holds all segmented positions for an experiment.  `segment_and_extract_features` runs StarDist nuclear segmentation on the chosen positions and extracts per-cell mean intensities for every channel, storing the result as a `PosLbl` pickle on disk.

Once segmented, load a position with:

```python
P = R.PosLbls[pos]   # P is a live reference — changes to P persist in R
```

`P` exposes:
- `P.channels` — list of channel names
- `P.frames` — time/z frames available
- `P.acq` — which acquisition this position belongs to
- `P.spatial` — dictionary of cached spatial results (populated as you run analyses below)

In [10]:
from oyLabImaging.Processing import results
R = results(MD=MD)

Loaded position C4-Site_0_0 from pickle file.
loaded results from pickle file


In [73]:
R.segment_and_extract_features(MD=MD,Position=['C4-Site_0_1', 'C4-Site_0_0'],NucChannel=R.channels[0],ffield=True,segment_type='stardist_nuclei')


Processing position C4-Site_0_1


100%|█████████████████████████████████████████████| 1/1 [00:58<00:00, 58.21s/it]


saved Position C4-Site_0_1
Finished loading and segmenting position C4-Site_0_1

Processing position C4-Site_0_0


100%|█████████████████████████████████████████████| 1/1 [01:05<00:00, 65.36s/it]


saved Position C4-Site_0_0
Finished loading and segmenting position C4-Site_0_0
saved Position C4-Site_0_0
saved results.
Loaded position C4-Site_0_0 from pickle file.

In [12]:
pos='C5-Site_0_0'
P = R.PosLbls[pos]
print('acquisition:', P.acq)
print('channels:', P.channels)
print('frames  :', P.frames)
print('ffield  :', P.framelabels[0]._ffield)

acquisition: confocal/Acq1_1
channels: ['DeepBlue' 'Yellow' 'FarRed']
frames  : [0]
ffield  : True


## Radial Correlation

**What it measures:** How correlated are cells in their expression as a function of distance?  For each pair of cells separated by distance r, we compute the Pearson correlation of their intensities and bin by r.  A value of 1 at short distance means nearby cells are perfectly co-expressed; a value near 0 means expression is spatially random.

**Two modes:**

- `img=False` (default) — cell-level: uses per-cell mean intensities from segmentation.  Each data point is a cell.  Distance resolution is limited by cell density (~5 µm bins typical).
- `img=True` — pixel-level: computes the autocorrelation of the raw image using FFT.  Much higher spatial resolution (~0.5 µm bins) but averages over all pixels, including cytoplasm and background.  Boundary artefacts are suppressed using periodic-plus-smooth decomposition (Moisan 2011).

**Interpreting the curve:**
- g(0) ≈ 1 for autocorrelation (same channel) — cells are perfectly correlated with themselves
- g(r) decays toward 0 with increasing r as cells become uncorrelated
- A long tail indicates large-scale spatial structure (e.g. tissue gradients or wave-like patterns)
- For cross-correlation (two different channels), g(0) reflects how co-expressed the channels are at the cell level; it can be negative if channels are anti-correlated

**Key parameters:**
- `max_r` — maximum distance to compute (µm); default 200
- `dr` — bin width (µm); default 5
- `ch_j` — second channel for cross-correlation; omit for autocorrelation

In [13]:
# Autocorrelation
res_auto = P.radial_corr('FarRed')
print('r range:', res_auto['r'][[0,-1]], 'µm')
print('g(0)   :', res_auto['g'][0])

r range: [  2.5 197.5] µm
g(0)   : 0.999999999999996


In [14]:
# Cross-correlation between two channels
res_cross = P.radial_corr('FarRed', 'Yellow')
print('g at r=0:', res_cross['g'][0])

g at r=0: -0.17150280317376085


In [15]:
# Plot autocorrelation + cross-correlation together
from oyLabImaging.Processing.spatial import plot_radial
ax = plot_radial([res_auto, res_cross], labels=['FarRed auto', 'FarRed×Yellow'])

In [17]:
P.plot_radial_corr('FarRed', ch_j=None, frame=None, img=False, ffield=True)

<Axes: xlabel='Distance (µm)', ylabel='Autocorrelation g(r)  [FarRed]'>

In [18]:
plt.close('all')

### Pixel-level radial correlation (`img=True`)

The pixel-level version uses the full fluorescence image rather than segmented cell centroids.  It captures sub-cellular spatial structure (e.g. nuclear vs cytoplasmic patterns) and is useful when you want finer distance resolution or do not have a reliable segmentation.  The trade-off is that it is sensitive to background and does not distinguish cell-to-cell variation from within-cell variation.

In [19]:
# Autocorrelation (img=True, ffield applied automatically)
res_img_auto = P.radial_corr('FarRed', img=True, max_r=250)
print('r range:', res_img_auto['r'][[0,-1]], 'µm')
print('g(0)   :', res_img_auto['g'][0])

r range: [  0.25 249.75] µm
g(0)   : 1.0035438277941013


In [20]:
# Cross-correlation between channels (img=True)
res_img_cross = P.radial_corr('FarRed', 'Yellow', img=True, max_r=250)
print('g at r=0:', res_img_cross['g'][0])

g at r=0: -0.04449315763417725


In [21]:
# Plot pixel-level results
ax = plot_radial([res_img_auto, res_img_cross],
                 labels=['FarRed auto (img)', 'FarRed×Yellow (img)'])

In [22]:
# Side-by-side: cell-level vs pixel-level autocorrelation
ax = plot_radial([res_auto, res_img_auto],
                 labels=['cell-level', 'pixel-level'])
ax.semilogy();

In [23]:
plt.close('all')

## Pair Correlation Function (Radial Density)

**What it measures:** Are cells spatially clustered, regularly spaced, or randomly distributed?  The pair correlation function g(r) is the ratio of the observed density of cell pairs at distance r to what you would expect from a completely random (Poisson) process.

**Interpretation:**
- g(r) = 1 — random arrangement at that distance (null expectation)
- g(r) > 1 — cells are more likely to be found at this separation than expected (clustering)
- g(r) < 1 — cells are less likely to be found at this separation (repulsion or exclusion zone)

A common pattern is g(r) > 1 at short distances (cells cluster in patches) followed by g(r) < 1 at slightly larger distances (spacing between patches), then returning to 1 at large r.

**Key parameters:**
- `max_r`, `dr` — distance range and bin width (µm)
- `n_max` — max pairs sampled per bin (for speed on large fields)

This method counts all cells regardless of channel.  It describes geometry, not expression.

In [24]:
res_density = P.radial_density()
print(' mean g(r):', res_density['g'].mean())

 mean g(r): 0.9237302074064413


In [25]:
ax = P.plot_radial_density()

In [26]:
plt.close('all')

## Correlation Lengthscale

**What it measures:** A single number summarising how far spatial correlation extends.  The radial correlation curve g(r) is fit to an exponential decay: `g(r) = A·exp(−r/λ) + C`, starting at `r_min` (to avoid the within-nucleus peak at very short distances).

**Output:**
- `λ` (lambda) — the e-folding length in µm.  Expression stays correlated over roughly λ µm.  Larger λ = longer-range coordination.
- `lambda_err` — 1σ uncertainty from the fit
- `r_sq` — R² of the fit; values > 0.95 indicate a clean exponential decay

**Biological interpretation:** λ reflects how far signalling or mechanical coupling acts.  A paracrine cytokine signal with short diffusion range might give λ ~ 20 µm (a few cell diameters); a long-range gradient might give λ ~ 100 µm.

**Key parameters:**
- `r_min` — starting distance for the fit (µm); should be at least one cell diameter to skip the within-cell contribution (typically 10–15 µm)

In [27]:
# Cell-level fit
pos='C5-Site_0_0'
P = R.PosLbls[pos]

fit_cell, ax = P.fit_corr_lengthscale('FarRed', r_min=10, plot=True)
print(f"λ = {fit_cell['lambda']:.1f} ± {fit_cell['lambda_err']:.1f} µm  "
      f"R² = {fit_cell['r_sq']:.3f}")

λ = 45.9 ± 0.8 µm  R² = 0.993


In [28]:
plt.close('all')
# Pixel-level fit (img=True)
fit_img, ax = P.fit_corr_lengthscale('FarRed', img=True, r_min=15, plot=True)
print(f"λ = {fit_img['lambda']:.1f} ± {fit_img['lambda_err']:.1f} µm  "
       f"R² = {fit_img['r_sq']:.3f}")

λ = 44.4 ± 0.0 µm  R² = 0.997


In [29]:
# Compare lengthscales across img or data (img=True)
ch='FarRed'
from oyLabImaging.Processing.spatial import plot_lengthscale_comparison
fits = [P.fit_corr_lengthscale(ch, img=im, r_min=10) for im in [True,False]]
ax = plot_lengthscale_comparison(fits, labels=['img','data'])
ax.set_ylim([30,60])

(30.0, 60.0)

In [30]:
plt.close('all')

## Mark Variogram

**What it measures:** The mark variogram is a complementary view of spatial correlation to the radial correlation function.  Instead of asking "how similar are nearby cells?", it asks "how different are they?".

For each pair of cells (i, j) separated by distance r, the normalised mark variogram is:

```
γ̃(r) = E[(m_i − m_j)²] / (2σ²_m)
```

where m is the expression mark (e.g. FarRed intensity) and σ is its standard deviation.  Dividing by 2σ² makes the reference value 1 (the expected value when marks are spatially random).

**Interpretation:**
- γ̃(r) < 1 — nearby cells are *more similar* than expected; positive spatial autocorrelation
- γ̃(r) = 1 — marks are spatially random at this distance (dashed reference line)
- γ̃(r) > 1 — nearby cells are *more different* than expected; negative autocorrelation (rare in biology)
- γ̃ starts low at short r and rises toward 1 as r increases and cells become uncorrelated

**Cross-variogram (two channels):** Provide a second channel to compute E[(A_i − A_j)(B_i − B_j)] / (2σ_A σ_B).  The reference line shifts to the global Pearson correlation between the two channels (the value at large r where spatial context is lost).  If the cross-variogram is *below* the reference at short r, nearby cells are more co-expressed than the global average — suggesting local coupling between the two signals.

**Key parameters:**
- `max_r`, `dr` — distance range and bin size (µm)
- `ch_j` — second channel for cross-variogram; omit for auto-variogram

In [32]:
# Auto mark variogram (single channel)
mv = P.mark_variogram('FarRed')
print(f"r range : {mv['r'][0]:.1f} – {mv['r'][-1]:.1f} µm")
print(f"γ̃ at smallest r : {mv['g'][0]:.3f}  (n={int(mv['n'][0])} pairs)")
print(f"γ̃ at largest r  : {mv['g'][-1]:.3f}  (ref = {mv['variogram_ref']:.3f})")

# Cross-variogram: FarRed × Yellow
mv_cross = P.mark_variogram('FarRed', 'Yellow')
print(f"\nCross-variogram FarRed × Yellow")
print(f"γ̃ ref (global Corr) : {mv_cross['variogram_ref']:.3f}")
print(f"γ̃ at smallest r     : {mv_cross['g'][0]:.3f}  (n={int(mv_cross['n'][0])} pairs)")

r range : 2.5 – 197.5 µm
γ̃ at smallest r : nan  (n=0 pairs)
γ̃ at largest r  : 0.959  (ref = 1.000)

Cross-variogram FarRed × Yellow
γ̃ ref (global Corr) : -0.172
γ̃ at smallest r     : nan  (n=0 pairs)


In [77]:
# Auto-variogram plot
ax = P.plot_mark_variogram('FarRed', img=False)

In [76]:
# Cross-channel comparison: all auto-variograms + cross-variogram on one plot
from oyLabImaging.Processing.spatial import plot_radial
mvs = [P.mark_variogram(ch,img=False) for ch in P.channels]
mv_cross = P.mark_variogram('FarRed', 'DeepBlue',img=False)
ax = plot_radial(mvs + [mv_cross],
                 labels=[f'{ch} auto' for ch in P.channels] + ['FarRed × Yellow'])

In [78]:
plt.close('all')

## LISA — Local Indicators of Spatial Association

**What it measures:** Local Moran's I identifies *which individual cells* are part of spatially autocorrelated clusters, rather than summarising the whole field with a single number.  For each cell i:

```
I_i = z_i · Σ_j w_ij z_j
```

where z is the z-scored expression and w_ij is a spatial weight (1 if j is within `radius` µm of i, else 0, Ripley-corrected at edges).  A high I_i means cell i is surrounded by cells that are similarly high or similarly low in expression.

**Interpretation:**
- High I, high expression → the cell is in a hot cluster (high-high)
- High I, low expression → the cell is in a cold cluster (low-low)
- Negative I → the cell is a spatial outlier (surrounded by dissimilar neighbours)

p-values come from permutation testing (the mark labels are shuffled and I_i recomputed many times).  Edge cells (within `radius` of the field boundary) have fewer neighbours and are flagged as unreliable.

**Visualisation:** `plot_lisa` opens a Napari window.  Points are coloured on a diverging coolwarm scale by their Moran's I value.  Cells with p > `max_pvalue` are dimmed (alpha=0.2) so significant clusters stand out.  Convex-hull outlines are drawn around DBSCAN clusters of significant cells.

**Key parameters:**
- `radius` — neighbourhood radius (µm); should match the expected cluster scale
- `n_permutations` — number of shuffles for p-value estimation (default 999)
- `max_pvalue` — alpha cutoff for display dimming and cluster detection
- `min_I`, `min_cells` — thresholds for calling discrete clusters

In [35]:
# Raw LISA result for one frame
lisa = P.local_moran_I('FarRed', frame=0, radius=30.0)
print('cells:', len(lisa['I']))
print('edge cells:', lisa['is_edge'].sum())
print('I range:', lisa['I'].min(), '-', lisa['I'].max())

cells: 18134
edge cells: 655
I range: -1.4554334878118351 - 78.68207285937353


In [36]:
# Find clusters from a pre-computed LISA result
clusts = P.find_activity_clusters(lisa, min_I=0.5, max_pvalue=0.001, min_cells=5)
print('clusters:', clusts['n_clusters'])
print('sizes   :', clusts['cluster_sizes'])

clusters: 20
sizes   : [171, 10, 6, 13, 26, 26, 13, 19, 47, 13, 7, 18, 11, 22, 22, 14, 28, 7, 6, 10]


In [37]:
# Combined: LISA + clustering in one call (no plotting)
cluster_results = P.activity_clusters(
    'FarRed', frame=0, radius=30.0,
    min_I=0.5, max_pvalue=0.001, min_cells=5
)
for res in cluster_results:
    print(f"frame {res['frame_index']}: {res['n_clusters']} clusters, "
          f"sizes {res['cluster_sizes']}")

frame 0: 20 clusters, sizes [171, 10, 6, 13, 26, 26, 13, 19, 47, 13, 7, 18, 11, 22, 22, 14, 28, 7, 6, 10]


In [40]:
# Napari visualisation: LISA + cluster overlays + raw channel image
layer, cluster_results = P.plot_lisa(
    'FarRed', frame=0,
    radius=30.0,
    min_I=0.5, max_pvalue=0.001, min_cells=5
)

## Getis-Ord Gi* — Hot and Cold Spot Detection

**What it measures:** Gi* tests whether the *total expression in a neighbourhood* is higher or lower than expected if the same values were randomly rearranged across the field.  For each cell i:

```
z_i = (Σ_j w_ij x_j − W_i·x̄) / s·√((n·Σ w²_ij − W²_i)/(n−1))
```

where x is expression, w_ij = 1 if j is within `radius` µm of i, W_i = Σ w_ij, and s is the global standard deviation.  p-values use the normal approximation — no permutation needed, so it is fast.

**Gi* vs LISA:**
- **LISA** asks: is *this cell* embedded in a cluster of similar cells?  Both the focal cell and its neighbours must be elevated.
- **Gi*** asks: is the neighbourhood of *this cell* collectively elevated?  The focal cell itself need not be unusual — only its surroundings.

Gi* is generally better at delineating the *extent* of hot/cold regions; LISA is more sensitive to detecting the *existence* of a cluster.

**Visualisation:** `plot_gistar` opens a Napari window with a coolwarm colourmap (red = hot, blue = cold).  Cells with p > `max_pvalue` are dimmed to alpha=0.2.  Hot spot clusters are outlined in pink, cold spot clusters in cyan.

**Key parameters:**
- `radius` — neighbourhood radius (µm)
- `max_pvalue` — significance cutoff for dimming and cluster detection (default 0.05)
- `min_z` — minimum |z-score| for cluster membership (default 1.96)
- `min_cells` — minimum cluster size (default 10)

In [41]:
# Compute Gi* for one frame
gs = P.gistar('FarRed', frame=0, radius=10.0)
print('cells     :', len(gs['z_score']))
print('edge cells:', gs['is_edge'].sum())
print('z range   :', gs['z_score'].min(), '-', gs['z_score'].max())
print('hot spots (z>1.96, p<0.05):', ((gs['z_score'] > 1.96) & (gs['pvalue'] < 0.05) & ~gs['is_edge']).sum())
print('cold spots (z<-1.96, p<0.05):', ((gs['z_score'] < -1.96) & (gs['pvalue'] < 0.05) & ~gs['is_edge']).sum())

cells     : 18134
edge cells: 268
z range   : -1.7984438202099966 - 18.237168805978186
hot spots (z>1.96, p<0.05): 711
cold spots (z<-1.96, p<0.05): 0


In [50]:
# Napari visualisation: red = hot clusters (pink outline), blue = cold clusters (cyan outline)
layer, cluster_results = P.plot_gistar('FarRed', frame=0, radius=30.0,
                                        max_pvalue=0.001, min_z=1.96, min_cells=5)
for fr in cluster_results:
    print(f"frame {fr['frame_index']}: "
          f"{fr['hot']['n_clusters']} hot clusters, "
          f"{fr['cold']['n_clusters']} cold clusters")

frame 0: 26 hot clusters, 0 cold clusters


## Spatial Regionalization

**What it measures:** Partitions the tissue into discrete spatial niches based on the *neighbourhood expression pattern* of multiple marker channels simultaneously.

For each cell, a **niche vector** is computed: the mean intensity of each marker channel within `radius` µm.  This captures not just what the cell expresses itself, but what its neighbours express — encoding the local microenvironment.  Niche vectors are z-score normalised across cells, then clustered.

The result is a tissue map where each region is defined by a distinct combination of locally elevated markers.  Region 1 might have cells surrounded by high FarRed and low DeepBlue neighbours; Region 2 might be the opposite.  This is fundamentally different from clustering individual cell expression, because a cell can belong to a region where *other nearby cells* are the ones expressing the relevant markers.

**Clustering methods:**
- `method='kmeans'` (default) — fast, hard assignment; good for well-separated regions
- `method='gmm'` — Gaussian mixture model; soft probabilistic assignment; handles overlapping regions better

**Number of regions:**
- Leave `n_regions=None` (default) to auto-select k from 2–`max_k` using the silhouette score
- Set `n_regions=k` to force a specific number

**Visualisation:** `plot_spatial_regions` opens a Napari window with cells coloured by region.  A summary table shows the mean expression of each marker per region, helping you interpret what each region represents biologically.

**Key parameters:**
- `channels` — list of marker channels to use for the niche vectors
- `radius` — neighbourhood radius (µm); controls the spatial scale of the niche
- `n_regions` — number of regions (None = auto)
- `method` — `'kmeans'` or `'gmm'`

In [51]:
# Auto-select number of regions (k=2..8 tested via silhouette)
res = P.spatial_regions(['DeepBlue', 'Yellow', 'FarRed'], radius=30.0)[0]
print(f"Auto-selected k={res['n_regions']}  silhouette={res['silhouette']:.3f}")
print(f"Cells per region: {[(res['labels']==k).sum() for k in range(res['n_regions'])]}")

Auto-selected k=3  silhouette=0.418
Cells per region: [9430, 7722, 982]


In [52]:
# Napari visualisation — colour by region, overlay all marker channels
layers = P.plot_spatial_regions(['DeepBlue', 'Yellow', 'FarRed'],
                                 radius=30.0, size=6)


Frame 0 — 3 regions (silhouette=0.418)
          DeepBlue  Yellow  FarRed
region 0     0.009   0.002   0.006
region 1     0.007   0.002   0.006
region 2     0.009   0.002   0.010


In [53]:
# Fix k=2 and use GMM instead of k-means
layers = P.plot_spatial_regions(['DeepBlue', 'Yellow', 'FarRed'],
                                 radius=20.0, n_regions=2, method='gmm', size=6)


Frame 0 — 2 regions (silhouette=0.349)
          DeepBlue  Yellow  FarRed
region 0     0.008   0.002   0.006
region 1     0.009   0.002   0.008


## Geographically Weighted Regression (GWR)

**What it measures:** Whether the relationship between two channels changes across the tissue.  A global correlation between channels A and B tells you whether they tend to be co-expressed, but hides the fact that the correlation might be strong in one region and absent (or even reversed) in another.

GWR fits a separate weighted least-squares regression at every cell, using a spatial kernel so that nearby cells contribute more to the local fit than distant ones:

```
FarRed_i ≈ β₀(i) + β₁(i)·Yellow_i + ...
```

The coefficients β are allowed to vary continuously across the tissue, producing a smooth map of where relationships are strong, weak, or reversed.

**Kernels:**
- `kernel='gaussian'` (default) — weights decay as exp(−d²/2bw²); cells beyond 3×bw contribute negligibly
- `kernel='bisquare'` — weights decay as (1−(d/bw)²)²; exactly zero beyond `bandwidth`; produces sharper spatial boundaries

**Bandwidth choice:** The `bandwidth` parameter (µm) is the most important tuning choice.  Too small → noisy maps with few neighbours per fit; too large → over-smoothed, missing real spatial variation.  A reasonable starting point is 2–5× the typical inter-cell distance.  Compare maps at a few bandwidths to assess robustness.

**Visualisation modes for `plot_gwr`:**
- `show='slope'` — local regression slope for a predictor channel; coolwarm colormap; dimmed where p > `max_pvalue`
- `show='r_squared'` — local fit quality (0–1); viridis; high values where the predictor explains the response well
- `show='intercept'` — local baseline expression of the response channel
- `show='t_stat'` — t-statistic of the slope; useful for comparing magnitude across positions

**Key parameters:**
- `ch_y` — response channel
- `ch_x` — predictor channel(s); can be a list for multiple regression
- `bandwidth` — kernel width (µm)
- `kernel` — `'gaussian'` or `'bisquare'`
- `predictor_idx` — which predictor to display when `ch_x` has multiple channels (0-indexed)

In [55]:
# Compute GWR: FarRed ~ Yellow (bandwidth = 30 µm, Gaussian kernel)
gwr_res = P.gwr('DeepBlue', 'FarRed', bandwidth=30.0)
r = gwr_res[0]
valid = ~np.isnan(r['r_squared'])
print(f"Cells with valid fit : {valid.sum()} / {len(valid)}")
print(f"Local R²  range     : {r['r_squared'][valid].min():.3f} – {r['r_squared'][valid].max():.3f}  "
      f"(median {np.median(r['r_squared'][valid]):.3f})")
slope = r['beta'][valid, 1]
print(f"Slope DeepBlue→FarRed : {slope.min():.3f} – {slope.max():.3f}  "
      f"(median {np.median(slope):.3f})")

Cells with valid fit : 18134 / 18134
Local R²  range     : 0.000 – 0.688  (median 0.042)
Slope DeepBlue→FarRed : -1.923 – 4.285  (median 0.413)


In [56]:
# Slope map: Yellow → FarRed, dim cells where p > 0.05
layer = P.plot_gwr('FarRed', 'DeepBlue', bandwidth=30.0, show='slope', max_pvalue=0.001)

In [60]:
# Local R² map — how well does DeepBlue predict FarRed here?
layer = P.plot_gwr('FarRed', 'DeepBlue', bandwidth=50.0, show='r_squared', max_pvalue=0.001)

saved Position C5-Site_0_0

In [61]:
plt.close('all')

## Batch Statistics and Cross-Position Report

The single-position methods above are useful for exploring individual fields.  When you have many positions (e.g. a full plate), use the batch workflow to compute statistics across all of them and compare.

### Step 1 — `R.calculate_spatial_stats()`

Runs the chosen statistics across a list of positions in parallel (default: all segmented positions).  Results are cached in each position's `P.spatial` dict **and** compiled into `R.spatial_stats`, which is persisted in the results pickle.

```
stats options:
  'radial_corr'    — radial correlation g(r) for each channel pair
  'radial_density' — pair correlation function (geometry only)
  'lengthscale'    — exponential fit λ; computed automatically with radial_corr
  'mark_variogram' — normalised mark variogram γ̃(r)
```

**Channel pairs computed:** autocorrelations (A×A) + all unique cross-pairs (A×B, not both A×B and B×A).

### Step 2 — `R.spatial_report()`

Generates matplotlib figures comparing statistics across positions.  Groups of positions are shown as mean ± SEM curves or bar heights, individual positions as single curves.

**Group resolution (in order of priority):**
1. Custom dict `groups={'label': [list of positions]}`
2. `group` field from the experiment metadata
3. Each position as its own group

In [83]:
# Compute all four stats for all segmented positions, using 4 parallel threads.
# Results land in R.spatial_stats and are also cached per-position in P.spatial.
R.calculate_spatial_stats(
    stats=['radial_corr', 'radial_density', 'lengthscale', 'mark_variogram'],
    ffield=True,
    n_jobs=4,img=False,
)

spatial stats: 100%|█████████████████████████████| 4/4 [00:11<00:00,  3.00s/pos]


{'radial_corr': {'C5-Site_0_1': {('DeepBlue',
    'DeepBlue'): {'r': array([  2.5,   7.5,  12.5,  17.5,  22.5,  27.5,  32.5,  37.5,  42.5,
            47.5,  52.5,  57.5,  62.5,  67.5,  72.5,  77.5,  82.5,  87.5,
            92.5,  97.5, 102.5, 107.5, 112.5, 117.5, 122.5, 127.5, 132.5,
           137.5, 142.5, 147.5, 152.5, 157.5, 162.5, 167.5, 172.5, 177.5,
           182.5, 187.5, 192.5, 197.5]), 'g': array([1.        , 0.50851835, 0.29153033, 0.24358361, 0.20951591,
           0.19398696, 0.18346247, 0.17087939, 0.15434221, 0.14756076,
           0.1450267 , 0.14062113, 0.12953565, 0.12519032, 0.12633052,
           0.11828323, 0.11536732, 0.11478566, 0.10602098, 0.10731373,
           0.1060432 , 0.10310193, 0.10316985, 0.10168743, 0.0968698 ,
           0.09546753, 0.09755251, 0.09522757, 0.09348089, 0.09024324,
           0.09422049, 0.09088798, 0.09312018, 0.09254739, 0.08248113,
           0.0818899 , 0.08550413, 0.08459167, 0.08525327, 0.08055929]), 'sem': array([0.02097115, 0

In [84]:
# Lengthscale summary — one row per position × channel pair
R.spatial_stats['lengthscale_summary']

,position,ch_i,ch_j,lam,lam_err,r_sq
0,C5-Site_0_1,DeepBlue,DeepBlue,3.785064e+01,9.840580e-01,0.977793
1,C5-Site_0_1,Yellow,Yellow,1.393439e+06,5.806532e+07,0.971911
2,C5-Site_0_1,FarRed,FarRed,4.476797e+01,7.565522e-01,0.993699
3,C5-Site_0_1,DeepBlue,Yellow,5.960000e+02,6.117804e+02,0.918852
4,C5-Site_0_1,DeepBlue,FarRed,7.325526e+01,1.244422e+01,0.910612
5,C5-Site_0_1,Yellow,FarRed,2.097654e+06,2.037948e+09,0.147104
6,C5-Site_0_0,DeepBlue,DeepBlue,4.487102e+01,1.589908e+00,0.964090
7,C5-Site_0_0,Yellow,Yellow,1.909282e+06,5.905525e+07,0.983932
8,C5-Site_0_0,FarRed,FarRed,4.588095e+01,8.058952e-01,0.993069
9,C5-Site_0_0,DeepBlue,Yellow,8.079433e+05,2.067289e+08,0.981873


In [85]:
# Report with each position as its own group (no grouping metadata in this experiment)
figs = R.spatial_report(groups={'C4': ['C4-Site_0_0', 'C4-Site_0_1'], 'C5': ['C5-Site_0_0', 'C5-Site_0_1']},channels=['FarRed'])

/home/alo649/Repos/oyLabImaging/oyLabImaging/Processing/Results.py:631: RuntimeWarning: Mean of empty slice
  mean_g = np.nanmean(g_mat, axis=0)
/home/alo649/.conda/envs/oyimg_tt/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1878: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [86]:
plt.close('all')